In [94]:
!python -m pip install -e ..

Obtaining file:///Users/dkoffical/Documents/GitHub/cs321m_project
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for torch_measure (pyproject.toml) ... done
  Created wheel for torch_measure: filename=torch_measure-0.1.1.dev64-0.editable-py3-none-any.whl size=3610 sha256=5dc904adaa9e1be53a7fa838302af4ed43d730ad6839f711bc4a6794e71d2f2c
  Stored in directory: /private/var/folders/qv/knpkl_fn1dx05nr8ny_7hnwm0000gn/T/pip-ephem-wheel-cache-0w639oq6/wheels/db/9f/66/061ff46a269df88e4649b7b397f9a65a8676ab6ac2ce02b09e
Successfully built torch_measure
  Attempting uninstall: torch_measure
    Found existing installation: torch_measure 0.1.1.dev61
    Uninstalling torch_measure-0.1.1.dev61:
      Successfully uninstalled torch_measure-0.1.1.dev61


In [95]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from torch_measure.datasets import load
from torch_measure.models import Rasch, TwoPL, ThreePL
from torch_measure.data import ResponseMatrix
from torch_measure.viz import plot_icc, plot_response_heatmap, plot_information
import torch_measure
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

import warnings
warnings.filterwarnings("ignore")

Using device: cpu


### Load a response matrix

Choose one matrix preset by changing `KFACTOR_MATRIX`. Rows are models/subjects, columns are items, and values are the observed score/correctness used by the factor model.


In [ ]:
import pandas as pd
import torch
from torch_measure.data import ResponseMatrix

MATRIX_PRESETS = {
    "mmlu_solver": {
        "prefix": "mmlu_pro_solver",
        "dir": "../benchmarks/mmlu/response_matrices",
        "benchmark_id": "mmlu_pro_solver",
        "item_content_field": "source",
        "item_id_field": "pair_id",
        "value": "correct: 1.0=true, 0.0=false, NaN=unparsed/missing",
    },
    "mmlu_judging": {
        "prefix": "mmlu_pro_judging",
        "dir": "../benchmarks/mmlu/response_matrices",
        "benchmark_id": "mmlu_pro_judging",
        "item_content_field": "source",
        "item_id_field": "pair_id/order",
        "value": "judge score/correctness; NaN=missing",
    },
    "safety_all_attacks": {
        "prefix": "safety_all_attacks",
        "dir": "../benchmarks/safety/final_solver/response_matrices",
        "benchmark_id": "safety_all_attacks",
        "item_content_field": "source",
        "item_id_field": "attack_family:input_index",
        "value": "score: 1.0=harmful/successful, 0.0=not harmful/unsuccessful, NaN=missing",
    },
    "code_solver": {
        "prefix": "livecodebench",
        "dir": "../benchmarks/code/response_matrices",
        "benchmark_id": "livecodebench",
        "item_content_field": "difficulty",
        "item_id_field": "question_id",
        "value": "correct: 1.0=passes public tests, 0.0=fails public tests, NaN=missing",
    },
    "code_judge": {
        "prefix": "codejudgebench_pairwise",
        "dir": "../benchmarks/code/response_matrices",
        "benchmark_id": "codejudgebench_pairwise",
        "item_content_field": "difficulty",
        "item_id_field": "split:question_id:pair_index:ordering",
        "value": "correct: 1.0=selected correct solution, 0.0=selected incorrect solution, NaN=unparsed",
    },
    "harmmetric_eval": {
        "prefix": "harmmetric_eval",
        "dir": "../benchmarks/HarmMetric_Eval/response_matrices",
        "benchmark_id": "harmmetric_eval",
        "item_content_field": "source",
        "item_id_field": "prompt_id",
        "value": "overall_effectiveness_score: soft/fractional score in [0, 1]",
    },
    "kudge_challenge": {
        "prefix": "kudge_challenge_easy_hard",
        "dir": "../benchmarks/kudge/results/kudge_challenge_easy_hard/response_matrices",
        "benchmark_id": "kudge_challenge_easy_hard",
        "item_content_field": "prompt",
        "item_id_field": "id",
        "value": "correct: 1.0=true, 0.0=false",
        "enrich_kudge": True,
    },
    "kudge_judge": {
        "prefix": "kudge_judge_easy_hard",
        "dir": "../benchmarks/kudge/results/kudge_judge_easy_hard/response_matrices",
        "benchmark_id": "kudge_judge_easy_hard",
        "item_content_field": "prompt",
        "item_id_field": "id",
        "value": "correct: 1.0=true, 0.0=false",
        "enrich_kudge": True,
    },
}

KFACTOR_MATRIX = "kudge_challenge"  # set by run_all_kfactor_notebooks.py
config = MATRIX_PRESETS[KFACTOR_MATRIX]
matrix_dir = config["dir"]
prefix = config["prefix"]

matrix_path = f"{matrix_dir}/{prefix}_response_matrix.csv"
subject_meta_path = f"{matrix_dir}/{prefix}_subject_metadata.csv"
item_meta_path = f"{matrix_dir}/{prefix}_item_metadata.csv"

print(f"Loading {KFACTOR_MATRIX}")
print(matrix_path)

df = pd.read_csv(matrix_path, index_col=0)
subject_meta = pd.read_csv(subject_meta_path)
item_meta = pd.read_csv(item_meta_path)

def enrich_kudge_item_metadata(item_meta):
    """Add prompt/source text from the original KUDGE dataset when available."""
    try:
        from datasets import load_dataset
    except ImportError:
        print("datasets is not installed; item metadata will only include response-matrix fields.")
        return item_meta

    try:
        ds = load_dataset("amphora/kudge-challenge", split="train")
    except Exception as exc:
        print(f"Could not load amphora/kudge-challenge; item metadata will only include response-matrix fields: {exc}")
        return item_meta

    source_rows = []
    for row in ds:
        if row.get("subset") not in {"Korean-Easy", "Korean-Hard"}:
            continue
        source_rows.append({
            "item_id": str(row.get("id")),
            "prompt": row.get("prompt"),
            "chosen": row.get("chosen"),
            "rejected": row.get("rejected"),
        })

    source_meta = pd.DataFrame(source_rows).drop_duplicates("item_id")
    enriched = item_meta.copy()
    enriched["item_id"] = enriched["item_id"].astype(str)
    enriched = enriched.merge(source_meta, on="item_id", how="left")
    print(f"Enriched item metadata with prompt text for {enriched['prompt'].notna().sum()}/{len(enriched)} items")
    return enriched

if config.get("enrich_kudge"):
    item_meta = enrich_kudge_item_metadata(item_meta)

item_content_field = config.get("item_content_field")
if item_content_field in item_meta.columns:
    item_contents = list(item_meta[item_content_field].fillna("").astype(str))
else:
    item_contents = list(item_meta.iloc[:, 0].astype(str))

rm = ResponseMatrix(
    data=torch.tensor(df.values, dtype=torch.float32),
    subject_ids=list(df.index.astype(str)),
    item_ids=list(df.columns.astype(str)),
    item_contents=item_contents,
    subject_metadata=subject_meta.to_dict("records"),
    info={
        "benchmark_id": config["benchmark_id"],
        "item_id_field": config["item_id_field"],
        "value": config["value"],
    },
)

print(f"{rm.n_subjects} models x {rm.n_items} items, density = {rm.density:.1%}")


Loading code_solver
../benchmarks/code/response_matrices/livecodebench_response_matrix.csv
9 models x 676 items, density = 100.0%


### K-Factor Fit

Fit `LogisticFM` with `K=1` and `K=2` on the response matrix. Rows stay as models/subjects and columns stay as items. Here `Z` is item easiness, so item difficulty is `-Z`.


In [97]:
import json
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm

from torch_measure.models import LogisticFM, predict_dense
from torch_measure.models.rotation import varimax_rotation
from torch_measure.metrics.functional import compute_all

# Shared dense matrix + observed mask for all K-factor fits.
data = rm.data.to(device).float()
mask = ~torch.isnan(data) & (data != -1)

print(f"K-factor data: {data.shape[0]} models x {data.shape[1]} items, observed={mask.float().mean().item():.1%}")


K-factor data: 9 models x 676 items, observed=100.0%


In [98]:
def make_heldout_split(mask, test_frac=0.2, seed=123):
    """Split observed response entries into train/eval masks."""
    observed = mask.nonzero(as_tuple=False)
    gen = torch.Generator(device="cpu").manual_seed(seed)
    perm = torch.randperm(observed.shape[0], generator=gen)
    n_eval = max(1, int(round(test_frac * observed.shape[0])))
    eval_idx = observed[perm[:n_eval]]

    train_mask = mask.clone()
    eval_mask = torch.zeros_like(mask, dtype=torch.bool)
    train_mask[eval_idx[:, 0], eval_idx[:, 1]] = False
    eval_mask[eval_idx[:, 0], eval_idx[:, 1]] = True
    return train_mask, eval_mask


def fit_logistic_fm_k(data, train_mask, k, device="cpu", max_epochs=1000, lr=0.03, seed=123, eval_mask=None, verbose=True):
    torch.manual_seed(seed + k)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed + k)

    model = LogisticFM(
        n_subjects=data.shape[0],
        n_items=data.shape[1],
        n_factors=k,
        device=device,
    )

    history = model.fit(
        data,
        mask=train_mask,
        method="mle",
        max_epochs=max_epochs,
        lr=lr,
        verbose=verbose,
    )

    with torch.no_grad():
        probs = predict_dense(model).detach()

    train_metrics = compute_all(probs, data, mask=train_mask)
    eval_metrics = compute_all(probs, data, mask=eval_mask) if eval_mask is not None else None

    return {
        "model": model,
        "history": history,
        "metrics": eval_metrics if eval_metrics is not None else train_metrics,
        "train_metrics": train_metrics,
        "eval_metrics": eval_metrics,
    }


kfactor_specs = {
    1: {"max_epochs": 1000, "lr": 0.03},
    2: {"max_epochs": 1000, "lr": 0.02},
}

HELDOUT_REPEATS = 5
HELDOUT_TEST_FRAC = 0.2

heldout_rows = []
for k, spec in kfactor_specs.items():
    for repeat in range(HELDOUT_REPEATS):
        split_seed = 123 + 1000 * repeat + k
        train_mask, eval_mask = make_heldout_split(mask, test_frac=HELDOUT_TEST_FRAC, seed=split_seed)
        print(f"\nHeld-out fit LogisticFM K={k}, repeat={repeat + 1}/{HELDOUT_REPEATS}")
        result = fit_logistic_fm_k(
            data,
            train_mask,
            k=k,
            device=device,
            max_epochs=spec["max_epochs"],
            lr=spec["lr"],
            seed=split_seed,
            eval_mask=eval_mask,
        )
        metrics = result["eval_metrics"]
        heldout_log_likelihood = metrics.get("log_likelihood")
        heldout_rows.append({
            "k": k,
            "repeat": repeat,
            "loss": -heldout_log_likelihood if heldout_log_likelihood is not None else None,
            "auc": metrics.get("auc"),
            "log_likelihood": heldout_log_likelihood,
            "brier": metrics.get("brier"),
            "ece": metrics.get("ece"),
        })

heldout_eval_raw = pd.DataFrame(heldout_rows)
heldout_eval_summary = (
    heldout_eval_raw
    .groupby("k", as_index=False)
    .agg(
        loss=("loss", "mean"),
        loss_std=("loss", "std"),
        auc=("auc", "mean"),
        auc_std=("auc", "std"),
        log_likelihood=("log_likelihood", "mean"),
        log_likelihood_std=("log_likelihood", "std"),
        brier=("brier", "mean"),
        brier_std=("brier", "std"),
        ece=("ece", "mean"),
        ece_std=("ece", "std"),
    )
)

# Final full-data fits are used for item difficulty estimates and exported tables.
kfactor_results = {}
for k, spec in kfactor_specs.items():
    print(f"\nFinal full-data fit LogisticFM K={k}")
    kfactor_results[k] = fit_logistic_fm_k(
        data,
        mask,
        k=k,
        device=device,
        max_epochs=spec["max_epochs"],
        lr=spec["lr"],
        seed=123,
    )

final_fit_summary = pd.DataFrame([
    {
        "k": k,
        "final_train_loss": result["history"]["losses"][-1],
        "final_train_auc": result["train_metrics"].get("auc"),
        "final_train_log_likelihood": result["train_metrics"].get("log_likelihood"),
        "final_train_brier": result["train_metrics"].get("brier"),
        "final_train_ece": result["train_metrics"].get("ece"),
    }
    for k, result in kfactor_results.items()
])

kfactor_fit_summary = heldout_eval_summary.merge(final_fit_summary, on="k", how="left")

display(kfactor_fit_summary.round(4))



Fitting LogisticFM K=1


MLE fitting: 100%|██████████| 1000/1000 [00:13<00:00, 76.22it/s, loss=0.110118]



Fitting LogisticFM K=2


MLE fitting: 100%|██████████| 1000/1000 [00:07<00:00, 137.60it/s, loss=0.059488]


,k,loss,auc,log_likelihood,brier,ece
0,1,0.1101,0.9985,-0.1101,0.0359,0.0187
1,2,0.0595,1.0000,-0.0595,0.0186,0.0130


### Item Difficulty With Laplace Uncertainty

For `LogisticFM`, `Z` is item easiness and `difficulty = -Z`. The uncertainty below uses a diagonal/conditional Laplace approximation: item factors and model factors are held fixed, and item-difficulty uncertainty is estimated from Bernoulli curvature `sum p(1-p)` over observed model responses.


In [99]:
def build_item_difficulty_table(result, rm, item_meta=None):
    model = result["model"]

    difficulty = model.difficulty.detach().cpu()
    difficulty_centered = difficulty - difficulty.mean()
    easiness_z = model.Z.detach().cpu()
    loadings = model.loadings.detach().cpu()

    # Rotate loadings for interpretability. For K=1 this is the identity.
    rotated_loadings, rotation = varimax_rotation(loadings)
    rotated_abilities = model.U.detach().cpu() @ rotation

    table = pd.DataFrame({
        "item_id": rm.item_ids,
        "difficulty": difficulty.numpy(),
        "difficulty_centered": difficulty_centered.numpy(),
        "easiness_z": easiness_z.numpy(),
    })

    for j in range(rotated_loadings.shape[1]):
        table[f"loading_factor_{j + 1}"] = rotated_loadings[:, j].numpy()

    loading_cols = [c for c in table.columns if c.startswith("loading_factor_")]
    table["dominant_factor"] = (
        table[loading_cols]
        .abs()
        .idxmax(axis=1)
        .str.replace("loading_factor_", "factor_", regex=False)
    )

    if item_meta is not None:
        table = table.merge(
            item_meta.reset_index(drop=True),
            left_index=True,
            right_index=True,
            how="left",
            suffixes=("", "_meta"),
        )

    model_factors = pd.DataFrame({"model": rm.subject_ids})
    for j in range(rotated_abilities.shape[1]):
        model_factors[f"factor_{j + 1}"] = rotated_abilities[:, j].numpy()

    return table, model_factors


item_difficulty_tables = {}
model_factor_tables = {}
for k, result in kfactor_results.items():
    item_table, model_table = build_item_difficulty_table(result, rm, item_meta=item_meta)
    item_difficulty_tables[k] = item_table
    model_factor_tables[k] = model_table

print("K=1 hardest items")
display(item_difficulty_tables[1].sort_values("difficulty_centered", ascending=False).head(20).round(3))

print("K=2 hardest items")
display(item_difficulty_tables[2].sort_values("difficulty_centered", ascending=False).head(20).round(3))


K=1 hardest items


,item_id,difficulty,difficulty_centered,easiness_z,loading_factor_1,dominant_factor,item_id_meta,selected_order,difficulty_meta,platform,question_content,starter_code,prompt
195,3558,6.645,6.557,-6.645,-3.605,factor_1,3558,195,medium,leetcode,NaN,NaN,NaN
7,2730,6.325,6.236,-6.325,-1.706,factor_1,2730,7,medium,leetcode,NaN,NaN,NaN
278,3777,5.957,5.869,-5.957,-3.411,factor_1,3777,278,hard,leetcode,NaN,NaN,NaN
30,2854,5.895,5.807,-5.895,-4.196,factor_1,2854,30,medium,leetcode,NaN,NaN,NaN
276,3771,5.822,5.734,-5.822,-3.788,factor_1,3771,276,medium,leetcode,NaN,NaN,NaN
365,abc324_d,5.774,5.685,-5.774,-4.834,factor_1,abc324_d,365,hard,atcoder,NaN,NaN,NaN
92,3230,5.683,5.594,-5.683,-3.019,factor_1,3230,92,medium,leetcode,NaN,NaN,NaN
187,3534,5.662,5.573,-5.662,-4.786,factor_1,3534,187,medium,leetcode,NaN,NaN,NaN
73,3166,5.658,5.569,-5.658,-2.430,factor_1,3166,73,medium,leetcode,NaN,NaN,NaN
56,3026,5.554,5.466,-5.554,-1.541,factor_1,3026,56,medium,leetcode,NaN,NaN,NaN


K=2 hardest items


,item_id,difficulty,difficulty_centered,easiness_z,loading_factor_1,loading_factor_2,dominant_factor,item_id_meta,selected_order,difficulty_meta,platform,question_content,starter_code,prompt
619,abc395_c,13.320,12.652,-13.320,1.195,5.471,factor_2,abc395_c,619,medium,atcoder,NaN,NaN,NaN
518,abc371_e,10.044,9.376,-10.044,0.901,4.050,factor_2,abc371_e,518,hard,atcoder,NaN,NaN,NaN
438,abc350_d,10.021,9.353,-10.021,0.898,4.040,factor_2,abc350_d,438,medium,atcoder,NaN,NaN,NaN
401,abc336_c,9.903,9.235,-9.903,0.529,7.764,factor_2,abc336_c,401,medium,atcoder,NaN,NaN,NaN
409,abc339_d,9.218,8.550,-9.218,-0.831,8.323,factor_2,abc339_d,409,medium,atcoder,NaN,NaN,NaN
582,abc386_b,9.117,8.449,-9.117,-0.736,8.032,factor_2,abc386_b,582,easy,atcoder,NaN,NaN,NaN
6,1899_D,8.630,7.962,-8.630,-0.770,7.745,factor_2,1899_D,6,hard,codeforces,NaN,NaN,NaN
364,abc324_c,8.441,7.772,-8.441,0.295,1.175,factor_2,abc324_c,364,medium,atcoder,NaN,NaN,NaN
450,abc354_a,8.250,7.582,-8.250,0.424,6.527,factor_2,abc354_a,450,easy,atcoder,NaN,NaN,NaN
345,abc318_c,8.134,7.466,-8.134,-0.830,7.512,factor_2,abc318_c,345,medium,atcoder,NaN,NaN,NaN


In [100]:
def item_difficulty_laplace_se(model, data, mask):
    """Conditional diagonal Laplace SE for LogisticFM item difficulty.

    This treats fitted model factors/loadings as fixed and estimates curvature
    only for each item intercept/difficulty. For difficulty = -Z, the sign
    does not change the curvature.
    """
    data = data.to(model.device).float()
    mask = mask.to(model.device)

    with torch.no_grad():
        probs = predict_dense(model).clamp(1e-7, 1 - 1e-7)
        info = torch.where(mask, probs * (1 - probs), torch.zeros_like(probs)).sum(dim=0)
        raw_se = 1 / torch.sqrt(info.clamp_min(1e-8))

        # Our main reported difficulty is mean-centered. Under a diagonal
        # approximation, propagate uncertainty through d_j - mean(d).
        n_items = raw_se.numel()
        raw_var = raw_se.pow(2)
        centered_var = ((1 - 1 / n_items) ** 2) * raw_var + (raw_var.sum() - raw_var) / (n_items ** 2)
        centered_se = torch.sqrt(centered_var.clamp_min(1e-12))

    return raw_se.detach().cpu(), centered_se.detach().cpu()


item_difficulty_with_uncertainty = {}
for k, result in kfactor_results.items():
    raw_se, centered_se = item_difficulty_laplace_se(result["model"], data, mask)

    table = item_difficulty_tables[k].copy()
    table["difficulty_laplace_se"] = raw_se.numpy()
    table["difficulty_centered_laplace_se"] = centered_se.numpy()
    table["difficulty_centered_laplace_lo"] = table["difficulty_centered"] - 1.96 * table["difficulty_centered_laplace_se"]
    table["difficulty_centered_laplace_hi"] = table["difficulty_centered"] + 1.96 * table["difficulty_centered_laplace_se"]
    item_difficulty_with_uncertainty[k] = table.sort_values("difficulty_centered", ascending=False).reset_index(drop=True)

print("K=1 item difficulties with Laplace uncertainty")
display(item_difficulty_with_uncertainty[1].round(3))

print("K=2 item difficulties with Laplace uncertainty")
display(item_difficulty_with_uncertainty[2].round(3))


K=1 item difficulties with Laplace uncertainty


,item_id,difficulty,difficulty_centered,easiness_z,loading_factor_1,dominant_factor,item_id_meta,selected_order,difficulty_meta,platform,question_content,starter_code,prompt,difficulty_laplace_se,difficulty_centered_laplace_se,difficulty_centered_laplace_lo,difficulty_centered_laplace_hi
0,3558,6.645,6.557,-6.645,-3.605,factor_1,3558,195,medium,leetcode,NaN,NaN,NaN,27.586000,27.546000,-47.433998,60.547001
1,2730,6.325,6.236,-6.325,-1.706,factor_1,2730,7,medium,leetcode,NaN,NaN,NaN,19.198000,19.171000,-31.337999,43.811001
2,3777,5.957,5.869,-5.957,-3.411,factor_1,3777,278,hard,leetcode,NaN,NaN,NaN,19.386999,19.358999,-32.075001,43.813000
3,2854,5.895,5.807,-5.895,-4.196,factor_1,2854,30,medium,leetcode,NaN,NaN,NaN,19.434000,19.406000,-32.230000,43.844002
4,3771,5.822,5.734,-5.822,-3.788,factor_1,3771,276,medium,leetcode,NaN,NaN,NaN,18.457001,18.431000,-30.392000,41.859001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
671,abc306_c,-12.659,-12.748,12.659,-4.790,factor_1,abc306_c,311,medium,atcoder,NaN,NaN,NaN,4.888000,4.886000,-22.323999,-3.171000
672,abc396_d,-13.505,-13.594,13.505,-5.099,factor_1,abc396_d,622,medium,atcoder,NaN,NaN,NaN,5.561000,5.558000,-24.487000,-2.700000
673,abc367_b,-13.931,-14.020,13.931,-2.664,factor_1,abc367_b,497,easy,atcoder,NaN,NaN,NaN,2.371000,2.378000,-18.681000,-9.358000
674,abc334_d,-14.989,-15.077,14.989,-2.865,factor_1,abc334_d,397,medium,atcoder,NaN,NaN,NaN,2.559000,2.566000,-20.106001,-10.048000


K=2 item difficulties with Laplace uncertainty


,item_id,difficulty,difficulty_centered,easiness_z,loading_factor_1,loading_factor_2,dominant_factor,item_id_meta,selected_order,difficulty_meta,platform,question_content,starter_code,prompt,difficulty_laplace_se,difficulty_centered_laplace_se,difficulty_centered_laplace_lo,difficulty_centered_laplace_hi
0,abc395_c,13.320,12.652,-13.320,1.195,5.471,factor_2,abc395_c,619,medium,atcoder,NaN,NaN,NaN,1.982,2.008,8.716000,16.587
1,abc371_e,10.044,9.376,-10.044,0.901,4.050,factor_2,abc371_e,518,hard,atcoder,NaN,NaN,NaN,1.604,1.637,6.167000,12.585
2,abc350_d,10.021,9.353,-10.021,0.898,4.040,factor_2,abc350_d,438,medium,atcoder,NaN,NaN,NaN,1.602,1.635,6.148000,12.558
3,abc336_c,9.903,9.235,-9.903,0.529,7.764,factor_2,abc336_c,401,medium,atcoder,NaN,NaN,NaN,2.640,2.658,4.025000,14.444
4,abc339_d,9.218,8.550,-9.218,-0.831,8.323,factor_2,abc339_d,409,medium,atcoder,NaN,NaN,NaN,2.380,2.400,3.845000,13.254
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
671,abc324_a,-9.697,-10.365,9.697,-1.303,0.178,factor_1,abc324_a,363,easy,atcoder,NaN,NaN,NaN,1.351,1.390,-13.090000,-7.640
672,abc325_e,-9.967,-10.635,9.967,-8.918,3.270,factor_1,abc325_e,370,hard,atcoder,NaN,NaN,NaN,1.606,1.638,-13.846000,-7.424
673,abc308_d,-10.528,-11.196,10.528,-9.502,3.491,factor_1,abc308_d,318,medium,atcoder,NaN,NaN,NaN,1.648,1.679,-14.487000,-7.905
674,abc306_e,-10.906,-11.574,10.906,-4.011,0.096,factor_1,abc306_e,312,hard,atcoder,NaN,NaN,NaN,5.221,5.224,-21.813999,-1.334


In [101]:
# Optional saves
result_name = KFACTOR_MATRIX
out_dir = Path("results") / result_name
out_dir.mkdir(parents=True, exist_ok=True)

def item_score_records(df, item_ids):
    """Return {item_id: {model: observed score}} with NaN converted to None."""
    score_map = {}
    for item_id in item_ids:
        values = df[str(item_id)] if str(item_id) in df.columns else df[item_id]
        score_map[str(item_id)] = {
            str(model): (None if pd.isna(value) else float(value))
            for model, value in values.items()
        }
    return score_map

scores_by_item = item_score_records(df, rm.item_ids)

kfactor_fit_summary.to_csv(out_dir / f"{result_name}_kfactor_fit_summary.csv", index=False)
item_difficulty_json = {
    "matrix": result_name,
    "benchmark_id": rm.info.get("benchmark_id"),
    "item_id_field": rm.info.get("item_id_field"),
    "value": rm.info.get("value"),
    "fits": {},
}
for k, table in item_difficulty_with_uncertainty.items():
    table.to_csv(out_dir / f"{result_name}_kfactor_k{k}_item_difficulties_with_laplace_uncertainty.csv", index=False)
    model_factor_tables[k].to_csv(out_dir / f"{result_name}_kfactor_k{k}_model_factors.csv", index=False)

    records = table.astype(object).where(pd.notna(table), None).to_dict("records")
    for record in records:
        record["scores"] = scores_by_item.get(str(record["item_id"]), {})
    item_difficulty_json["fits"][f"k{k}"] = records

with open(out_dir / f"{result_name}_kfactor_item_difficulties_with_laplace_uncertainty.json", "w") as f:
    json.dump(item_difficulty_json, f, indent=2)

print(f"Saved K-factor tables to {out_dir.resolve()}")


Saved K-factor tables to /Users/dkoffical/Documents/GitHub/cs321m_project/K-Factor/results/code_solver


In [102]:
item_difficulty_json['fits']

{'k1': [{'item_id': '3558',
   'difficulty': 6.6452460289001465,
   'difficulty_centered': 6.556765079498291,
   'easiness_z': -6.6452460289001465,
   'loading_factor_1': -3.604956865310669,
   'dominant_factor': 'factor_1',
   'item_id_meta': '3558',
   'selected_order': 195,
   'difficulty_meta': 'medium',
   'platform': 'leetcode',
   'question_content': None,
   'starter_code': None,
   'prompt': None,
   'difficulty_laplace_se': 27.586109161376953,
   'difficulty_centered_laplace_se': 27.546213150024414,
   'difficulty_centered_laplace_lo': -47.43381118774414,
   'difficulty_centered_laplace_hi': 60.54734420776367,
   'scores': {'claude_haiku_4_5': 0.0,
    'mistral_3b': 0.0,
    'mistral_8b': 0.0,
    'mistral_14b': 0.0,
    'qwen3_5_0_8b': 0.0,
    'qwen3_5_2b': 0.0,
    'qwen3_5_4b': 0.0,
    'qwen3_5_9b': 0.0,
    'qwen3_5_27b': 0.0}},
  {'item_id': '2730',
   'difficulty': 6.32481050491333,
   'difficulty_centered': 6.236329555511475,
   'easiness_z': -6.32481050491333,
   'l